# Generate Music Embeddings on Google Colab

This notebook generates semantic embeddings for music plays using state-of-the-art sentence transformers.

**Models available:**
- `all-mpnet-base-v2` (420M params, 768 dims) - Best quality/speed tradeoff
- `all-MiniLM-L12-v2` (33M params, 384 dims) - Faster, smaller
- `multi-qa-mpnet-base-dot-v1` (420M params, 768 dims) - Optimized for semantic search

**Requirements:**
1. Upload `enriched_plays_full.csv` to Colab
2. Enable GPU runtime (Runtime → Change runtime type → T4 GPU)
3. Run all cells

In [ ]:
# Install dependencies
!pip install -q sentence-transformers pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import time
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.17 GB


In [ ]:
import shutil
import os

def save_to_gdrive(file_path, destination_dir='/content/drive/MyDrive/music/'):
    """Saves a file to Google Drive."""
    if not os.path.exists(destination_dir):
        os.makedirs(destination_dir, exist_ok=True)
        print(f"Created destination directory: {destination_dir}")

    destination_path = os.path.join(destination_dir, os.path.basename(file_path))
    shutil.copy(file_path, destination_path)
    print(f"Successfully copied '{file_path}' to '{destination_path}'")

In [ ]:
# Configuration
INPUT_FILE = 'enriched_plays_full.csv'  # Upload this file first
# MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'  # Best quality
MODEL_NAME = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'  # Alternative: optimized for search
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L12-v2'  # Alternative: faster/smaller

BATCH_SIZE = 1024 if device == 'cuda' else 32  # Larger batches for GPU
OUTPUT_EMBEDDINGS = 'embeddings.npy'
OUTPUT_METADATA = 'metadata.csv'

print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")

Model: sentence-transformers/multi-qa-mpnet-base-dot-v1
Batch size: 1024


In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Load data
print("Loading data...")
df = pd.read_csv('/content/drive/MyDrive/music/enriched_plays_full.csv')

print(f"Loaded {len(df):,} plays")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample enriched text:")
print(df['enriched_text'].iloc[0][:200] + "...")

# Extract texts for embedding
texts = df['enriched_text'].fillna('').tolist()
print(f"\nPrepared {len(texts):,} texts for embedding")

Loading data...


/tmp/ipython-input-2952606236.py:3: DtypeWarning: Columns (4,7,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/music/enriched_plays_full.csv')


Loaded 2,193,235 plays

Columns: ['id', 'artist', 'artist_ids', 'song', 'recording_id', 'album', 'release_id', 'release_group_id', 'release_date', 'labels', 'label_ids', 'airdate', 'rotation_status', 'is_local', 'is_request', 'is_live', 'comment', 'show', 'enriched_text']

Sample enriched text:
Ami Taf Ra feat. Kamasi Washington - How I Became a Madman - The Prophet and the Madman | Comment: North African, LA-based singer-songwriter Ami Taf Ra has announced her debut album, The Prophet and T...

Prepared 2,193,235 texts for embedding


In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)

embedding_dim = model.get_sentence_embedding_dimension()
print(f"✓ Model loaded")
print(f"  Embedding dimensions: {embedding_dim}")
print(f"  Max sequence length: {model.max_seq_length}")

Loading model: sentence-transformers/multi-qa-mpnet-base-dot-v1...
✓ Model loaded
  Embedding dimensions: 768
  Max sequence length: 512


In [ ]:
# Generate embeddings
print(f"\nGenerating embeddings for {len(texts):,} texts...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

elapsed = time.time() - start_time
texts_per_sec = len(texts) / elapsed

print(f"\n✓ Generated {len(embeddings):,} embeddings")
print(f"  Time: {elapsed:.1f}s ({texts_per_sec:.1f} texts/sec)")
print(f"  Shape: {embeddings.shape}")
print(f"  Size: {embeddings.nbytes / 1e6:.1f} MB")

save_to_gdrive(OUTPUT_EMBEDDINGS)


Generating embeddings for 2,193,235 texts...
Batch size: 1024
Device: cuda


Batches:   0%|          | 0/2142 [00:00<?, ?it/s]


✓ Generated 2,193,235 embeddings
  Time: 2875.2s (762.8 texts/sec)
  Shape: (2193235, 768)
  Size: 6737.6 MB


FileNotFoundError: [Errno 2] No such file or directory: 'embeddings.npy'

2193235

In [ ]:
# Save embeddings
print(f"\nSaving embeddings to {OUTPUT_EMBEDDINGS}...")
np.save(OUTPUT_EMBEDDINGS, embeddings)
print(f"✓ Saved embeddings ({embeddings.nbytes / 1e6:.1f} MB)")
save_to_gdrive(OUTPUT_EMBEDDINGS)

# Save metadata (subset of columns for efficient loading)
metadata_columns = [
    'id', 'artist', 'artist_ids', 'song', 'recording_id',
    'album', 'release_id', 'airdate', 'labels', 'rotation_status'
]
available_columns = [col for col in metadata_columns if col in df.columns]

print(f"\nSaving metadata to {OUTPUT_METADATA}...")
df[available_columns].to_csv(OUTPUT_METADATA, index=False)
print(f"✓ Saved metadata ({len(available_columns)} columns)")
save_to_gdrive(OUTPUT_METADATA)

print(f"\n{'='*80}")
print("COMPLETE!")
print(f"{'='*80}")
print(f"\nGenerated files:")
print(f"  1. {OUTPUT_EMBEDDINGS} - Numpy array ({embeddings.shape[0]:,} x {embeddings.shape[1]})")
print(f"  2. {OUTPUT_METADATA} - Metadata CSV ({len(df):,} rows)")
print(f"\nDownload both files and use them in your application!")


Saving embeddings to embeddings.npy...
✓ Saved embeddings (6737.6 MB)
Successfully copied 'embeddings.npy' to '/content/drive/MyDrive/music/embeddings.npy'

Saving metadata to metadata.csv...
✓ Saved metadata (10 columns)
Successfully copied 'metadata.csv' to '/content/drive/MyDrive/music/metadata.csv'

COMPLETE!

Generated files:
  1. embeddings.npy - Numpy array (2,193,235 x 768)
  2. metadata.csv - Metadata CSV (2,193,235 rows)

Download both files and use them in your application!


## Test Search (Optional)

Test the embeddings with a sample search query:

In [ ]:
# Test search
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=10):
    """Search for similar music using semantic similarity."""
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    # Compute similarities (already normalized, so dot product = cosine similarity)
    similarities = embeddings @ query_embedding.T

    # Get top results
    top_indices = np.argsort(similarities.flatten())[-top_k:][::-1]

    print(f"\nQuery: '{query}'\n")
    print(f"{'='*80}")

    for rank, idx in enumerate(top_indices, 1):
        score = similarities[idx][0]
        play = df.iloc[idx]
        print(f"{rank:2d}. [{score:.3f}] {play['artist']} - {play['song']}")
        if 'album' in play and pd.notna(play['album']):
            print(f"    Album: {play['album']}")
        print()

# Example searches
search("psychedelic folk rock")
search("upbeat dance electronic")
search("melancholic indie with female vocals")


Query: 'psychedelic folk rock'

 1. [0.734] Kaleidoscope - Oh Death
    Album: Side Trips

 2. [0.734] Kaleidoscope - Oh Death
    Album: Side Trips

 3. [0.733] Kaleidoscope - Oh Death
    Album: Side Trips

 4. [0.733] Kaleidoscope - Oh Death
    Album: Side Trips

 5. [0.733] Kaleidoscope - Oh Death
    Album: Side Trips

 6. [0.730] Kaleidoscope - Oh Death
    Album: Side Trips

 7. [0.693] ID - ID

 8. [0.693] ID - ID

 9. [0.692] ID - ID

10. [0.692] ID - ID


Query: 'upbeat dance electronic'

 1. [0.718] Electronic - Feel Every Beat
    Album: Electronic

 2. [0.698] Technotronic - Pump Up The Jam
    Album: New Beat Take 5

 3. [0.688] Electronic - Getting Away With It
    Album: Get the Message: The Best of Electronic

 4. [0.680] Electronic - Get the Message
    Album: Electronic

 5. [0.679] Technotronic - Pump Up the Jam (Crowd Is Jumping mix)
    Album: Best of Technotronic

 6. [0.678] Electronic - Get the Message
    Album: Electronic

 7. [0.678] Electronic - Get the M

In [ ]:
# Colab Notebook: Dimensionality Reduction
import numpy as np
from sklearn.decomposition import PCA
import umap

# Load your embeddings
embeddings = np.load('/content/music/embeddings.npy')  # (2193235, 768)

# PCA to 256 dimensions (3x smaller, ~95% variance retained)
pca = PCA(n_components=256, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

# Save
np.save('embeddings_256d.npy', embeddings_pca)
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.3f}")

# OPTIONAL: UMAP for even better clustering (slower)
# reducer = umap.UMAP(n_components=128, random_state=42)
# embeddings_umap = reducer.fit_transform(embeddings[:100000])  # Sample first

# Benefits:
# - 768 dims → 256 dims = 3x less storage (6.7GB → 2.2GB)
# - 3x faster similarity search
# - ~95% of semantic information retained

Variance retained: 0.964


In [ ]:
import joblib

# Define a filename for your PCA model
PCA_MODEL_FILE = 'pca_transformer_256d.joblib'

print(f"Saving PCA transformer to {PCA_MODEL_FILE}...")

# Save the 'pca' object to a file
joblib.dump(pca, PCA_MODEL_FILE)

print(f"✓ Saved PCA transformer.")

# --- Save to Google Drive ---
# (This assumes your 'save_to_gdrive' function is defined)
try:
    save_to_gdrive(PCA_MODEL_FILE)
except NameError:
    print("\nWarning: 'save_to_gdrive' function not defined in this session.")
    print("Run the cell containing 'save_to_gdrive' if you want to copy this to Drive.")
except Exception as e:
    print(f"\nAn error occurred while saving to Google Drive: {e}")

Saving PCA transformer to pca_transformer_256d.joblib...
✓ Saved PCA transformer.
Successfully copied 'pca_transformer_256d.joblib' to '/content/drive/MyDrive/music/pca_transformer_256d.joblib'


In [ ]:
# Colab Notebook: Clustering
from sklearn.cluster import MiniBatchKMeans
from sentence_transformers import SentenceTransformer
import pandas as pd

# Load embeddings
embeddings = np.load('embeddings.npy')
metadata = pd.read_csv('metadata.csv')

# Cluster into 100 semantic groups
kmeans = MiniBatchKMeans(n_clusters=100, batch_size=10000, random_state=42)
clusters = kmeans.fit_predict(embeddings)

# Add to metadata
metadata['cluster_id'] = clusters

# Generate cluster labels using representative samples
model = SentenceTransformer('all-MiniLM-L6-v2')

cluster_labels = {}
for cluster_id in range(100):
    # Get most central point
    cluster_mask = clusters == cluster_id
    cluster_embeddings = embeddings[cluster_mask]
    center = kmeans.cluster_centers_[cluster_id]

    # Find 5 closest samples
    distances = np.linalg.norm(cluster_embeddings - center, axis=1)
    closest_indices = np.argsort(distances)[:5]

    # Get their texts (assuming you have text in metadata)
    sample_texts = metadata[cluster_mask].iloc[closest_indices]['text'].tolist()

    # Label the cluster (you could also use an LLM here)
    cluster_labels[cluster_id] = f"cluster_{cluster_id}"  # Or use GPT to generate names

metadata['cluster_label'] = metadata['cluster_id'].map(cluster_labels)
metadata.to_csv('metadata_enriched.csv', index=False)


In [ ]:
len(embeddings_pca)

2193235

# Task
Refine the music embeddings clustering process by updating the metadata saving to include 'enriched_text', re-running the metadata saving, and then modifying the clustering code to use the 'enriched_text' for generating more descriptive cluster labels, using the same SentenceTransformer model, and saving the `metadata_enriched.csv` to Google Drive.

## Modify metadata saving to include enriched_text

### Subtask:
The current `metadata.csv` does not include the 'enriched_text' column, which is crucial for generating meaningful LLM-based cluster labels. This step will modify cell `bjTwZMuWNqrn` to include 'enriched_text' when saving the `metadata.csv` file.


**Reasoning**:
The subtask requires modifying the `metadata_columns` list in cell `bjTwZMuWNqrn` to include 'enriched_text' before saving `metadata.csv`. I will update the code in that cell to reflect this change.

